# Cortex Agents for Multiple Tenants/Users
There are a number of use cases where we want to use the Cortex AI features of Snowflake but only give certain
users access to certain data in the database. 

For example, we may have sales data for our company, and we want to create an application that lets our salespeople 
ask questions in a "talk-to-your-data" sort of fashion, but we only want a salesperson to get access to the data about 
their customers and not other customers. Another example is that we have an application that manages data for many 
tenants, and we want to allow a tenant to ask questions of their data, but not get any access to other tenants' data.
Lastly, we may have a situation where users of our applicaiton also have access to Snowflake (say, via an SSO integration)
and we want to give users access to data as if they logged into Snowflake themselves.

We can accomplish this using Snowflake, leveraging the Cortex Agents feature for AI and the Row Access Policy
(RAP) feature to restrict the data for the user.

This Notebook will walk you through the setup for a 3-tier web app that uses Cortex Agents to talk to your data
and manages sales data for multiple tenants. Users will log in to the application and only see results related
to the tenant they are allowed to. The use case is managing restaurant sales data (how much each menu item
is selling at each restaurant within each chain). 

The high-level steps in the Notebook are:
* set up a warehouse, database, and schema for this example
* create a role for the example and grant it the necessary privileges to create the objects it needs
* create a set of menu items (using Cortex to generate descriptions)
* create a Cortex Search service for the menu items
* create daily sales totals for each store in each restaurant
* create a Programmatic Access Token (PAT) to allow our application to talk to Snowflake APIs
* create a Row Access Policy and apply it to the items and orders tables
* print the environment variables that you will need to run the companion web app

## Example Use Case
The example that we consider is a website that helps restaurant franchise owners analyze their sales
data. Franchises upload their menu items and descriptions to the service (we'll store that in the
`ITEMS` table). Additionally, franchises send their sales data to this service, and that data is summarized
for daily sales (we'll store that in the `ORDERS` table). Each franchise has multiple stores, and the sales data 
is captured per store and per item. Just for illustration, the data in this demo contains sales data just for
the calendar year 2024.

The website allows for a franchise owner to log in and ask natuarl language questions about their menu
and sales data. The franchise owner should only be able to see the data about their franchise. For exmaple,
Alice, owner of Alices Restaurant, can analyze data about all of the Alices Restaurant stores, and cannot
ask questions about any other franchise, such as Bobs Place or Charlies Diner.

Some of the types of questions that an owner might ask are:
* _Which chicken sandwich sold the most in May 2024?_
* _Which menu items that are not chicken sandwiches sold the least in April 2024?_
* _Did we sell more sandwiches or burgers in June 2024?_
* _Which menu item sold the most for all of 2024?_

## Row Access Policies
There are actually 2 distinct scenarios we will consider with respect to how we limit data access.

1. Users of our application also have a login to Snowflake via an External OAuth integration with an Identity
   Provider, and those users should see data just like if they logged into Snowflake and issued SQL themselves.
   We will consider an entitlement table that maps a user to the tenants they are authorized to view.

2. Users of our application will log into the applicaiton via an Identity Provider who will return a tenant
   key that can be used to restrict the access to only specific tenants as specified in an entitlement table.
   We actually have 2 different approaches to this:

   1. We can use a Snowflake role-per-tenant model. In this case, we will have a table that maps from the tenant
      key to the role name. Then the application will use that role when calling Snowflake. The entitlement table
      that specified which tenant role can access which tenants is protected by a Row Access Policy based on the
      `CURRENT_ROLE`.
   
   2. We can use a session variable approach. In this case, we will use an entitlement table that maps from the
      tenant key to the tenants that the tenant key can access. We will use a Row Access Policy that uses the value
      of a session variable to restrict the tenant values in the entitlement table.

This Notebook will walk you through the steps to set this up. You can choose which approach to apply for your scenario.

## 1. Setup
First, we will create the `WAREHOUSE` that we can use for this project, as well as a 
`DATABASE` and `SCHEMA` to host the objects.

In [ ]:
%%sql -r dataframe_1
USE ROLE accountadmin;
CREATE WAREHOUSE IF NOT EXISTS multisales_wh WITH WAREHOUSE_SIZE='XSMALL';
CREATE DATABASE IF NOT EXISTS multisales;
CREATE SCHEMA IF NOT EXISTS multisales.data;

Next, we create the `MULTISALES_RL` role that will own the objects and grant it the necessary permissions.

In [ ]:
%%sql -r dataframe_2
USE ROLE accountadmin;
CREATE ROLE IF NOT EXISTS multisales_rl;
GRANT ROLE multisales_rl TO ROLE accountadmin;
SET cur_user = CURRENT_USER();
GRANT ROLE multisales_rl TO USER Identifier($cur_user);

GRANT USAGE ON WAREHOUSE multisales_wh TO ROLE multisales_rl;
GRANT ALL ON DATABASE multisales TO ROLE multisales_rl;
GRANT ALL ON SCHEMA multisales.data TO ROLE multisales_rl;
GRANT DATABASE ROLE snowflake.cortex_user TO ROLE multisales_rl;
GRANT CREATE ROLE ON ACCOUNT TO ROLE multisales_rl;

For the scenario where we will talk to Snowflake using a service user, we want to create a role for that user's permissions, called `MULTISALES_APP_RL`.
We will grant the `MULTISALES_RL` to the `MULTISALES_APP_RL`.

In [ ]:
%%sql -r dataframe_7
CREATE ROLE IF NOT EXISTS multisales_app_rl;
GRANT ROLE multisales_app_rl TO ROLE accountadmin;
GRANT ROLE multisales_rl TO ROLE multisales_app_rl;

In [ ]:
%%sql -r dataframe_16
USE ROLE multisales_rl;
USE SCHEMA multisales.data;

## 2. Data
Next, we create the data for our example.

### Items
Now, we create the menu items for our restaurants. There are 3 restaurant chains:
* Alices Restaurant
* Bobs Place
* Charlies Diner

They sell a variety of sandwiches, burgers, pasta, and tacos. We will use Cortex to generate the descriptions of 
the menu items based on their item names.

In [ ]:
%%sql -r dataframe_3
CREATE OR REPLACE
TABLE items AS
    WITH items AS (
        SELECT 101::INT AS item_id, 'Alices Restaurant' AS tenant, 'Chicken Salad Sandwich' AS item
        UNION ALL
        SELECT 102::INT AS item_id, 'Alices Restaurant' AS tenant, 'Chicken Caesar Wrap' AS item
        UNION ALL
        SELECT 103::INT AS item_id, 'Alices Restaurant' AS tenant, 'Grilled Chicken Sandwich' AS item
        UNION ALL
        SELECT 104::INT AS item_id, 'Alices Restaurant' AS tenant, 'Cheeseburger' AS item
        UNION ALL
        SELECT 105::INT AS item_id, 'Alices Restaurant' AS tenant, 'Bacon Double Cheeseburger' AS item
        UNION ALL
        SELECT 106::INT AS item_id, 'Alices Restaurant' AS tenant, 'Swiss Bleu Burger' AS item
        UNION ALL
        SELECT 107::INT AS item_id, 'Alices Restaurant' AS tenant, 'Spaghetti Bolognese' AS item
        UNION ALL
        SELECT 108::INT AS item_id, 'Alices Restaurant' AS tenant, 'Pesto Tortellini' AS item
        UNION ALL
    
        SELECT 201::INT AS item_id, 'Bobs Place' AS tenant, 'Fried Chicken Sandwich' AS item
        UNION ALL
        SELECT 202::INT AS item_id, 'Bobs Place' AS tenant, 'Chicken Bacon Ranch Sandwich' AS item
        UNION ALL
        SELECT 203::INT AS item_id, 'Bobs Place' AS tenant, 'Hamburger' AS item
        UNION ALL
        SELECT 204::INT AS item_id, 'Bobs Place' AS tenant, 'Cheddar Burger' AS item
        UNION ALL
        SELECT 205::INT AS item_id, 'Bobs Place' AS tenant, 'Swiss Burger' AS item
        UNION ALL
        SELECT 206::INT AS item_id, 'Bobs Place' AS tenant, 'Chicken Tacos' AS item
        UNION ALL
        SELECT 207::INT AS item_id, 'Bobs Place' AS tenant, 'Beef Tacos' AS item
        UNION ALL
        SELECT 208::INT AS item_id, 'Bobs Place' AS tenant, 'Veggie Tacos' AS item
        UNION ALL
    
        SELECT 301::INT AS item_id, 'Charlies Diner' AS tenant, 'Smoked Chicken Sandwich' AS item
        UNION ALL
        SELECT 302::INT AS item_id, 'Charlies Diner' AS tenant, 'Korean Fried Chicken Sandwich' AS item
        UNION ALL
        SELECT 303::INT AS item_id, 'Charlies Diner' AS tenant, 'Greek Chicken Wrap' AS item
        UNION ALL
        SELECT 304::INT AS item_id, 'Charlies Diner' AS tenant, 'Fried Fish Sandwich' AS item
        UNION ALL
        SELECT 305::INT AS item_id, 'Charlies Diner' AS tenant, 'Tuna Salad Sandwich' AS item
    )
    SELECT 
        item_id,
        tenant,
        item,
        AI_COMPLETE('claude-4-sonnet', 'You are writing menu descriptions for the following menu item: ' || item || '. The description should be no longer than 50 words. The description should entice customers to order the item')::VARCHAR AS item_description
    FROM items;

ALTER TABLE items SET CHANGE_TRACKING = TRUE;

### Orders
Next, we create the daily sales totals for each menu item at each store for each restaurant franchise.

In [ ]:
%%sql -r dataframe_4
CREATE OR REPLACE TABLE orders AS 
    WITH days AS (
        SELECT 
            DATEADD(day, seq4(), '2024-01-01'::DATE) AS date
        FROM TABLE (GENERATOR(ROWCOUNT => 365))
    )
    , tenant_popularity AS (
        SELECT 'Alices Restaurant' AS tenant, 500::INT AS popularity
        UNION ALL
        SELECT 'Bobs Place' AS tenant, 100::INT AS popularity
        UNION ALL
        SELECT 'Charlies Diner' AS tenant, 1000::INT AS popularity
    )
    , item_popularity AS (
        SELECT 
            i.item_id,
            i.item,
            tp.tenant,
            tp.popularity + NORMAL(0, 100, RANDOM(123)) AS popularity
        FROM items AS i
        JOIN tenant_popularity AS tp
          ON i.tenant = tp.tenant
    )
    , stores AS (
        SELECT 1001+seq4() AS store_id, 'Alices Restaurant' AS tenant
        FROM TABLE(GENERATOR(ROWCOUNT => 4))
        UNION ALL
        SELECT 2001+seq4() AS store_id, 'Bobs Place' AS tenant
        FROM TABLE(GENERATOR(ROWCOUNT => 3))
        UNION ALL
        SELECT 3001+seq4() AS store_id, 'Charlies Diner' AS tenant
        FROM TABLE(GENERATOR(ROWCOUNT => 5))
    )
    , orders AS (
        SELECT 
            d.date AS date,
            s.store_id,
            s.tenant,
            ip.item_id AS item_id,
            ip.item AS item,
            FLOOR(ip.popularity + NORMAL(0,100, RANDOM(234))) AS quantity
        FROM stores AS s
        JOIN item_popularity AS ip
          ON s.tenant = ip.tenant
        CROSS JOIN days AS d
    )
    SELECT
        * 
    FROM orders
;

### Cortex Search Services
We want to leveage Cortex Search to help Cortex Agents understand our questions and enable better
accuracy for answers. We will create two Cortex Search Services on the `ITEMS` table: one on the `ITEM` column and one 
on the `ITEM_DESCRIPTION` column.

In [ ]:
%%sql -r dataframe_5
CREATE OR REPLACE CORTEX SEARCH SERVICE items_search
    ON item
    ATTRIBUTES item_id, tenant, item_description
    WAREHOUSE = multisales_wh
    TARGET_LAG = '1 minute'
    AS (
        SELECT
            item_id,
            tenant,
            item,
            item_description
        FROM items
    )
;

CREATE OR REPLACE CORTEX SEARCH SERVICE item_description_search
    ON item_description
    ATTRIBUTES item_id, tenant, item
    WAREHOUSE = multisales_wh
    TARGET_LAG = '1 minute'
    AS (
        SELECT
            item_id,
            tenant,
            item,
            item_description
        FROM items
    )
;

### Semantic View
We will need a semantic model for this project. We could use either a semantic model in a file in `STAGE`, 
or a `SEMANTIC VIEW`. For this exmaple, let's use a `SEMANTIC VIEW`.

In [ ]:
%%sql -r dataframe_6
CREATE OR REPLACE SEMANTIC VIEW multisales_sv
	tables (
		ITEMS primary key (ITEM_ID) comment='This table stores information about items across multiple tenants, with each item having a unique identifier, a descriptive name, and a detailed description.',
		ORDERS comment='This table stores information about orders placed across different stores and tenants, including the date of the order, the store ID, the tenant name, the item ID and name, and the quantity of the item ordered.'
	)
	relationships (
		ORDERS_TO_ITEMS as ORDERS(ITEM_ID) references ITEMS(ITEM_ID)
	)
	facts (
		ORDERS.QUANTITY as QUANTITY comment='The quantity of items ordered.'
	)
	dimensions (
		ITEMS.ITEM as ITEM comment='This column represents the name of the menu item, which is a dimension used to categorize and analyze sales data.' with cortex search service ITEMS_SEARCH,
		ITEMS.ITEM_DESCRIPTION as ITEM_DESCRIPTION comment='A detailed description of each menu item, including ingredients, preparation methods, and flavor profiles, designed to entice customers and provide a clear understanding of the dish.',
		ITEMS.ITEM_ID as ITEM_ID comment='Unique identifier for each item in the inventory.',
		ITEMS.TENANT as TENANT comment='The name of the tenant or business that occupies a specific location or space.',
		ORDERS.DATE as DATE comment='Date the order was placed.',
		ORDERS.ITEM as ITEM comment='The type of menu item ordered by the customer.',
		ORDERS.ITEM_ID as ITEM_ID comment='Unique identifier for the item being ordered.',
		ORDERS.STORE_ID as STORE_ID comment='Unique identifier for the store where the order was placed.',
		ORDERS.TENANT as TENANT comment='The name of the tenant or customer who placed the order.'
	)
	comment='This semantic model contains menu items and daily sales for multiple restaurant chains and stores for business analysis.'
	with extension (CA='{"tables":[{"name":"ITEMS","dimensions":[{"name":"ITEM","sample_values":["Chicken Salad Sandwich","Chicken Caesar Wrap","Veggie Tacos"]},{"name":"ITEM_DESCRIPTION","sample_values":["Tender, hand-pulled chicken mixed with crisp celery, fresh herbs, and creamy mayo, nestled between slices of artisan bread. This classic comfort sandwich delivers the perfect balance of flavors and textures, making it an irresistible choice for lunch that will leave you completely satisfied.","Juicy, perfectly seasoned chicken breast grilled to golden perfection and nestled on a toasted brioche bun with crisp lettuce, ripe tomato, and creamy mayo. Each bite delivers tender, smoky flavors that will satisfy your cravings. A classic done right – simple, fresh, and absolutely delicious.","**Cheddar Burger**\\n\\nJuicy, flame-grilled beef patty topped with rich, melted aged cheddar cheese on a toasted brioche bun. Served with crisp lettuce, ripe tomato, and red onion. This classic comfort food delivers bold, satisfying flavors in every bite. Pure indulgence for cheese lovers craving the perfect burger experience."]},{"name":"ITEM_ID","sample_values":["101","102","103"]},{"name":"TENANT","sample_values":["Bobs Place","Charlies Diner","Alices Restaurant"]}]},{"name":"ORDERS","dimensions":[{"name":"ITEM","sample_values":["Chicken Salad Sandwich","Chicken Caesar Wrap","Grilled Chicken Sandwich"]},{"name":"ITEM_ID","sample_values":["101","102","103"]},{"name":"STORE_ID","sample_values":["1003","1001","3001"]},{"name":"TENANT","sample_values":["Alices Restaurant"]}],"facts":[{"name":"QUANTITY","sample_values":["449","317","588"]}],"time_dimensions":[{"name":"DATE","sample_values":["2024-01-01","2024-01-03","2024-01-02"]}]}],"relationships":[{"name":"orders_to_items"}],"module_custom_instructions":{}}')
;

### Agent
Now we create the `MULTISALES_AGENT`.

In [ ]:
%%sql -r dataframe_18
CREATE OR REPLACE AGENT multisales_agent
    FROM SPECIFICATION 
$$
{
    "models": {
        "orchestration":"auto"
    },
    "orchestration": {},
    "instructions": {
        "sample_questions":[
            {"question": "List the available tenants and the number of store locations they have."},
            {"question": "What were the best selling items in April 2024?"},
            {"question": "What was the best selling item over all time?"},
            {"question": "What was the best selling chicken sandwich in May 2024?"}
        ]
    },
    "tools": [
        {
            "tool_spec": {
                "type": "cortex_analyst_text_to_sql",
                "name": "analyst1","description": "Use this tool to query menu and sales data for various tenants."
            }
        },
        {
            "tool_spec": {
                "type": "cortex_search",
                "name": "search1",
                "description": "Indexes the menu item names"
            }
        }
    ],
    "tool_resources": {
        "analyst1": {
            "execution_environment": {
                "type": "warehouse",
                "warehouse": "MULTISALES_WH"
            },
            "semantic_view" :"MULTISALES.DATA.MULTISALES_SV"
        },
        "search1": {
            "id_column": "ITEM_ID",
            "max_results": 10,
            "search_service": "MULTISALES.DATA.ITEMS_SEARCH",
            "title_column": "ITEM"
        }
    }
}
$$
;

## 3. Row Access Policy
Now we turn our attention to protecting the data so that a given user can only access the tenant data they are entitled to.

There are 3 different setups that each have different types of entitlement tables and Row Access Policies that we will consider. 
* **3a:** SSO where users will have direct access to Snowflake and our entitlement can be based on their Snowflake username (`CURRENT_USER()`).
* **3b:** App users only have access to some tenants. The app will talk to Snowflake using a Snowflake role and our entitlement can be based
  on that role (`CURRENT_ROLE()`).
* **3c:** App users only have access to some tenants. The app will set a session variable when talking to Snowflake and our entitlement can be
  based on the value of that variable (`GETVARIABLE('TENANT')`).


We'll show how to do each of these below, but in reality you should **choose one of the options that matches your use case best**.

In all cases, we will use a _memoizable_ User Defined Function to make an `ARRAY` of tenant values from the entitlement table for the 
given user.


### 3a. SSO
The entitlement table for this scenario will have 2 columns: Snowflake username and tenant identifier. The Row Access Policy
will restrict rows from this entitlement table based on the value of `CURRENT_USER()` and the Snowflake username column.

In [ ]:
%%sql -r dataframe_8
CREATE OR REPLACE TABLE entitlement_sso(username VARCHAR, tenant VARCHAR);

-- We'll add entitlement to all 3 tenant for the current user.
-- You can also experiment by removing some of these entries.
INSERT INTO entitlement_sso(username, tenant) VALUES
    (CURRENT_USER(), 'Alices Restaurant'),
    (CURRENT_USER(), 'Bobs Place'),
    (CURRENT_USER(), 'Charlies Diner')
;

-- RAP based on CURRENT_USER()
CREATE OR REPLACE ROW ACCESS POLICY rap_entitlement_sso
    AS (username VARCHAR)
    RETURNS BOOLEAN ->
        username = CURRENT_USER()
;

-- Set RAP on entitlement table based on the `username` column.
ALTER TABLE entitlement_sso DROP ALL ROW ACCESS POLICIES; -- Just to make sure there aren't any others
ALTER TABLE entitlement_sso ADD ROW ACCESS POLICY rap_entitlement_sso ON (username);

-- Create a memoizable UDF that builds the tenant array
CREATE OR REPLACE FUNCTION tenants_sso()
    RETURNS ARRAY
    MEMOIZABLE
    AS 'SELECT ARRAY_AGG(tenant) FROM entitlement_sso'
;

-- Create a RAP to apply to the Orders and Items tables
CREATE OR REPLACE ROW ACCESS POLICY rap_tenant_sso
    AS (tenant VARCHAR)
    RETURNS BOOLEAN ->
        ARRAY_CONTAINS(tenant::VARIANT, tenants_sso())
;

In [ ]:
%%sql -r dataframe_9
-- Apply the RAP to the Orders and Items tables
ALTER TABLE orders DROP ALL ROW ACCESS POLICIES; -- Just to make sure there aren't any
ALTER TABLE orders ADD ROW ACCESS POLICY rap_tenant_sso ON (tenant);

ALTER TABLE items DROP ALL ROW ACCESS POLICIES; -- Just to make sure there aren't any
ALTER TABLE items ADD ROW ACCESS POLICY rap_tenant_sso ON (tenant);

### 3b. Role-per-Tenant
In this case, the application will set the current role to the tenant role before calling Snowflake, so the entitlement
table here will be based on the role name and `CURRENT_ROLE()`.

In [ ]:
-- We need to create the roles for the tenants. We will grant those roles to `MULTISALES_RL`.
CREATE ROLE IF NOT EXISTS RL_ALICE;
CREATE ROLE IF NOT EXISTS RL_BOB;
CREATE ROLE IF NOT EXISTS RL_CHARLIE;
GRANT ROLE RL_ALICE TO ROLE multisales_app_rl;
GRANT ROLE RL_BOB TO ROLE multisales_app_rl;
GRANT ROLE RL_CHARLIE TO ROLE multisales_app_rl;

-- We need to grant those roles access to the ORDERS and ITEMS tables
GRANT ROLE multisales_rl TO ROLE rl_alice;
GRANT ROLE multisales_rl TO ROLE rl_bob;
GRANT ROLE multisales_rl TO ROLE rl_charlie;

-- We need a table to help the application know how to map from the tenant key to the tenant role.
CREATE OR REPLACE TABLE tenant_roles(tenant_key VARCHAR, tenant_role VARCHAR);
INSERT INTO tenant_roles(tenant_key, tenant_role) VALUES
    ('Alice', 'RL_ALICE'),
    ('Bob', 'RL_BOB'),
    ('Charlie', 'RL_CHARLIE')
;

-- Now create the entitlement table.
CREATE OR REPLACE TABLE entitlement_role(tenant_role VARCHAR, tenant VARCHAR);

-- We'll add entitlement for each tenant role to the associated tenant.
INSERT INTO entitlement_role(tenant_role, tenant) VALUES
    ('RL_ALICE', 'Alices Restaurant'),
    ('RL_BOB', 'Bobs Place'),
    ('RL_CHARLIE', 'Charlies Diner')
;

-- RAP based on CURRENT_USER()
CREATE OR REPLACE ROW ACCESS POLICY rap_entitlement_role
    AS (tenant_role VARCHAR)
    RETURNS BOOLEAN ->
        CASE WHEN 'ACCOUNTADMIN' = CURRENT_ROLE() THEN TRUE
        ELSE UPPER(tenant_role) = UPPER(CURRENT_ROLE()) END
;

-- Set RAP on entitlement table based on the `tenant_role` column.
ALTER TABLE entitlement_role DROP ALL ROW ACCESS POLICIES; -- Just to make sure there aren't any others
ALTER TABLE entitlement_role ADD ROW ACCESS POLICY rap_entitlement_role ON (tenant_role);

-- Create a memoizable UDF that builds the tenant array
CREATE OR REPLACE FUNCTION tenants_role()
    RETURNS ARRAY
    MEMOIZABLE
    AS 'SELECT ARRAY_AGG(tenant) FROM entitlement_role'
;

-- Create a RAP to apply to the Orders and Items tables
CREATE OR REPLACE ROW ACCESS POLICY rap_tenant_role
    AS (tenant VARCHAR)
    RETURNS BOOLEAN ->
        ARRAY_CONTAINS(tenant::VARIANT, tenants_role())
;

In [ ]:
%%sql -r dataframe_11
-- Apply the RAP to the Orders and Items tables
ALTER TABLE orders DROP ALL ROW ACCESS POLICIES; -- Just to make sure there aren't any
ALTER TABLE orders ADD ROW ACCESS POLICY rap_tenant_role ON (tenant);

ALTER TABLE items DROP ALL ROW ACCESS POLICIES; -- Just to make sure there aren't any
ALTER TABLE items ADD ROW ACCESS POLICY rap_tenant_role ON (tenant);

### 3c. Session Variable
In this case, the application will set the session variable `TENANT` before calling Snowflake, so the entitlement
table here will be based on the tenant_key and `GETVARIABLE('TENANT')`. The application will also use the `MULTISALES_RL` 
role when it makes calls, so our Row Access Policy will apply only when the `CURRENT_ROLE()` is `MULTISALES_RL`.

In [ ]:
%%sql -r dataframe_12
CREATE OR REPLACE TABLE entitlement_var(tenant_key VARCHAR, tenant VARCHAR);

-- We'll add entitlement for each tenant role to the associated tenant.
INSERT INTO entitlement_var(tenant_key, tenant) VALUES
    ('Alice', 'Alices Restaurant'),
    ('Bob', 'Bobs Place'),
    ('Charlie', 'Charlies Diner')
;

-- RAP based on CURRENT_USER()
CREATE OR REPLACE ROW ACCESS POLICY rap_entitlement_var
    AS (tenant_key VARCHAR)
    RETURNS BOOLEAN ->
        CASE
            WHEN 'MULTISALES_RL' = CURRENT_ROLE() THEN tenant_key = GETVARIABLE('TENANT')
            ELSE TRUE
        END
;

-- Set RAP on entitlement table based on the `tenant_role` column.
ALTER TABLE entitlement_var DROP ALL ROW ACCESS POLICIES; -- Just to make sure there aren't any others
ALTER TABLE entitlement_var ADD ROW ACCESS POLICY rap_entitlement_var ON (tenant_key);

-- Create a memoizable UDF that builds the tenant array
CREATE OR REPLACE FUNCTION tenants_var()
    RETURNS ARRAY
    MEMOIZABLE
    AS 'SELECT ARRAY_AGG(tenant) FROM entitlement_var'
;

-- Create a RAP to apply to the Orders and Items tables
CREATE OR REPLACE ROW ACCESS POLICY rap_tenant_var
    AS (tenant VARCHAR)
    RETURNS BOOLEAN ->
        CASE
            WHEN 'MULTISALES_RL' = CURRENT_ROLE() THEN ARRAY_CONTAINS(tenant::VARIANT, tenants_var())
            ELSE TRUE
        END
;

In [ ]:
%%sql -r dataframe_13
-- Apply the RAP to the Orders and Items tables
ALTER TABLE orders DROP ALL ROW ACCESS POLICIES; -- Just to make sure there aren't any
ALTER TABLE orders ADD ROW ACCESS POLICY rap_tenant_var ON (tenant);

ALTER TABLE items DROP ALL ROW ACCESS POLICIES; -- Just to make sure there aren't any
ALTER TABLE items ADD ROW ACCESS POLICY rap_tenant_var ON (tenant);

## 4. Service User and Programmatic Access Token
In cases **3b** and **3c**, the application will need to talk to Snowflake using a service user. We will configure this to use a Programmatic
Access token for simplicity. You could use other approaches to connect to Snowflake as a service user, such as keypair-based JWTs (see Snowflake documentation).

### Service User
Make sure to change the password in the following command to something appropriate (and secure).

In [ ]:
USE ROLE accountadmin;
CREATE USER IF NOT EXISTS multisales_app 
    TYPE = SERVICE
    DEFAULT_ROLE = 'MULTISALES_RL'
    DEFAULT_NAMESPACE = 'MULTISALES.DATA'
    DEFAULT_WAREHOUSE = 'MULTISALES_WH'
    DEFAULT_SECONDARY_ROLES = () 
;

-- Grant it the roles it needs
GRANT ROLE multisales_rl TO USER multisales_app;
GRANT ROLE multisales_app_rl TO USER multisales_app;

### Programmatic Access Token
Now we will create a Programmatic Access Token for our service user.

In [ ]:
-- We need to create and apply an Authentication Policy that allows us to have a PAT token with no role restriction.
CREATE AUTHENTICATION POLICY IF NOT EXISTS pat_anyrole_policy
  PAT_POLICY = (
    REQUIRE_ROLE_RESTRICTION_FOR_SERVICE_USERS = FALSE
  );

ALTER USER multisales_app SET AUTHENTICATION POLICY pat_anyrole_policy;

-- Now create the PAT token
ALTER USER multisales_app ADD PROGRAMMATIC ACCESS TOKEN multisales_pat;

In [ ]:
%%sql -r dataframe_20
-- Just to be consistent, let's go back to using the MULTISALES_RL
USE ROLE multisales_rl;

## 5. Companion App
There is a React frontend component and Express server component that implements these patterns in this repository and show how to add
a talk-to-your-data interface to an existing application, powered by Snowflake Cortex. There is also a sample application that uses 
these components to show a complete application.

In all three cases, you will need to configure an Identity Provider to manage access to the application. Check the READMEs and the
example environment files for more information on configuring that.

In all cases, copy the `sample-app/env.frontend.example` file to `sample-app/.env.local` and the `sample-app/env.backend.example`
file to `sample-app/.env`. 

There is some common configuration in all cases:

#### Frontend (`.env.local`) configuration
* `VITE_AUTH_MODE` to `OAUTH`
* Identity Provider
  * `VITE_OAUTH_LOGIN_URL` per your Identity Provider
  * `VITE_OAUTH_CLIENT_ID` per your Identity Provider

#### Backend (`.env`) configuration
* Snowflake
  * `SNOWFLAKE_HOST` to your Snowflake hostname (should be in the form of `ORGNAME-ACCTNAME.snowflakecomputing.com`).
  * `SNOWFLAKE_DATABASE` to `MULTISALES`.
  * `SNOWFLAKE_SCHEMA` to `DATA`.
* Identity Provider
  * `OAUTH_TOKEN_URL` per your Identity Provider
  * `OAUTH_CLIENT_ID` per your Identity Provider
  * `OAUTH_CLIENT_SECRET` per your Identity Provider

We will show how to configure each scenario below.

### 5a. SSO

#### Frontend (`.env.local`)
Set the following variables in the `.env.local` file:
* Identity Provider
  * `VITE_OAUTH_SCOPE` to `session:role-any`

#### Backend (`.env`)
Set the following variables in the `.env` file:
* `AUTH_MODE` to `OAUTH`


### 5b. Role-per-Tenant

#### Frontend (`.env.local`)
* Identity Provider
  * `VITE_OAUTH_SCOPE` to `openid profile email`
  * `VITE_OAUTH_AUDIENCE` per your Identity Provider

#### Backend (`.env`)
* `AUTH_MODE` to `HYBRID`
* Snowflake
  * `SNOWFLAKE_PAT` to the PAT we created above.
  * `SNOWFLAKE_WAREHOUSE` to `MULTISALES_WH`
  * `SNOWFLAKE_ROLE` to `MULTISALES_RL`
* Identity Provider
  * `IDP_JWKS_URL` per your Identity Provider
  * `IDP_ISSUER` per your Identity Provider
  * `IDP_AUDIENCE` per your Identity Provider
  * `CLAIM_KEY` to `tenant_key` (or whatever you configure your Identity Provider claim to be the tenant)
  * `USERNAME_CLAIM_KEY` to `email` (or whatever claim key we should use for the user to isolate their threads in Cortex)
* Multitenancy
  * `TENANT_ISOLATION_MODE` to `ROLE`
  * `TENANT_ROLE_TABLE` to `MULTISALES.DATA.TENANT_ROLES`


### 5c. Session Variable

#### Frontend (`.env.local`)
* Identity Provider
  * `VITE_OAUTH_SCOPE` to `openid profile email`
  * `VITE_OAUTH_AUDIENCE` per your Identity Provider

#### Backend (`.env`)
* `AUTH_MODE` to `HYBRID`
* Snowflake
  * `SNOWFLAKE_PAT` to the PAT we created above.
* Identity Provider
  * `IDP_JWKS_URL` per your Identity Provider
  * `IDP_ISSUER` per your Identity Provider
  * `IDP_AUDIENCE` per your Identity Provider
  * `CLAIM_KEY` to `tenant_key` (or whatever you configure your Identity Provider claim to be the tenant)
  * `USERNAME_CLAIM_KEY` to `email` (or whatever claim key we should use for the user to isolate their threads in Cortex)
* Multitenancy
  * `TENANT_ISOLATION_MODE` to `SESSION_VAR`
  * `SESSION_VAR_NAME` to `TENANT`

